# Honey Yield Prediction

## DuckDB Practice and Feature Engineering

Author: Stephanie Nord

# 1. Project Setup

This section establishes the project environment by connecting to the DuckDB database, defining reusable project paths, and locating the source data. Organizing these tasks at the beginning of the notebook makes the workflow portable and easier to reproduce.

## Connect to DuckDB

```
Zenodo ZIP
      │
      ▼
Extract Daily Measurement Files
      │
      ▼
DuckDB (honey_daily)
      │
      ▼
Data Quality Assessment
      │
      ▼
Transformation
      │
      ▼
Feature Engineering
      │
      ▼
Modeling Dataset
      │
      ▼
Predictive Model
```

In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import zipfile

In [2]:
# Connect to DuckDB
con = duckdb.connect("honey.duckdb")

In [3]:
# Define project paths
working_dir = Path.cwd()
project_root = working_dir.parent

print(f"Working Directory: {working_dir}")
print(f"Project Root: {project_root}")

Working Directory: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Stephanie_Work
Project Root: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model


In [4]:
list(project_root.iterdir())

[WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/.env'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/.git'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/.gitignore'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/.ipynb_checkpoints'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Beehive_Honey_Yield_Project.ipynb'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/data-guide.md'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/David_Work'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/honey-yield-predictive-model'),
 WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yie

# 2. Extract Data

This section locates the publication archive, inspects its contents, and extracts the daily measurement files used throughout the remainder of the project.

```text
Zenodo Publication Archive (.zip)
        │
        ▼
Inspect Archive
        │
        ▼
Identify Daily Measurement Files
        │
        ▼
Extract Daily Measurement Files

In [5]:
# Locate ZIP files in Downloads
downloads = Path.home() / "Downloads"

zip_files = list(downloads.glob("*.zip"))
zip_files

[WindowsPath('C:/Users/sfnor/Downloads/61c0b904b3c4831e6578e564.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive (1).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive (2).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive (3).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive (4).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive (5).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/archive.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/Beltmatic.v1.0.7 (1).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/Beltmatic.v1.0.7.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/bob_publication_data.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/crystal_quest_vscode.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/cursorblade_demo_v0.27f (1).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/cursorblade_demo_v0.27f.zip'),
 WindowsPath('C:/Users/sfnor/Downloads/data-visualization-exercise (1).zip'),
 WindowsPath('C:/Users/sfnor/Downloads/data-visualization-exercise.zip'),
 WindowsPath('C:/Users/sf

In [6]:
# Select the Zenodo publication dataset ZIP
zip_path = downloads / "bob_publication_data.zip"

zip_path.exists(), zip_path

(True, WindowsPath('C:/Users/sfnor/Downloads/bob_publication_data.zip'))

In [7]:
# Inspect ZIP contents without extracting
with zipfile.ZipFile(zip_path) as z:
    files = z.namelist()

len(files)

1865

> **Design Decision**
>
> The publication archive contains multiple data products, including event files and multiple temporal resolutions. Only the daily measurement files are extracted because they provide an appropriate starting point for exploratory analysis and predictive modeling.

In [8]:
# Preview first 50 files in the archive
files[:50]

['bob_publication_data/',
 'bob_publication_data/readme.txt',
 'bob_publication_data/years/',
 'bob_publication_data/years/2019/',
 'bob_publication_data/years/2019/preprocessed/',
 'bob_publication_data/years/2019/preprocessed/2019_d/',
 'bob_publication_data/years/2019/preprocessed/2019_d/11.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/22.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/25.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/26.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/27.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/43.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/46.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/48.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/49.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/5.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/56.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/58.csv',
 'bob_p

In [9]:
# Search for files or folders related to daily data
daily_files = [f for f in files if "day" in f.lower() or "daily" in f.lower()]

len(daily_files), daily_files[:50]

(0, [])

In [10]:
daily_files = [f for f in files if "_d_" in f]

len(daily_files)

0

In [11]:
daily_files[:20]

[]

In [12]:
# Find all daily files
daily_files = [f for f in files if "_d" in f]

len(daily_files)

1865

In [13]:
daily_files[:20]

['bob_publication_data/',
 'bob_publication_data/readme.txt',
 'bob_publication_data/years/',
 'bob_publication_data/years/2019/',
 'bob_publication_data/years/2019/preprocessed/',
 'bob_publication_data/years/2019/preprocessed/2019_d/',
 'bob_publication_data/years/2019/preprocessed/2019_d/11.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/22.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/25.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/26.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/27.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/43.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/46.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/48.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/49.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/5.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/56.csv',
 'bob_publication_data/years/2019/preprocessed/2019_d/58.csv',
 'bob_p

In [14]:
# Count all 3 data resolutions at once
daily_files = [f for f in files if "_d" in f]
hourly_files = [f for f in files if "_h" in f]
minute_files = [f for f in files if "_m" in f]

print(f"Daily files:  {len(daily_files)}")
print(f"Hourly files: {len(hourly_files)}")
print(f"Minute files: {len(minute_files)}")

Daily files:  1865
Hourly files: 350
Minute files: 632


In [15]:
con.execute("""
SELECT COUNT(*) AS row_count
FROM honey_daily;
""").df()

,row_count
0,36697106


In [16]:
measurement_files = [
    f for f in files
    if "years" in f
    and "preprocessed" in f
    and "_d" in f
    and "events" not in f
    and f.endswith(".csv")
]

len(measurement_files), measurement_files[:20]

(453,
 ['bob_publication_data/years/2019/preprocessed/2019_d/11.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/22.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/25.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/26.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/27.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/43.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/46.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/48.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/49.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/5.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/56.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/58.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/6.csv',
  'bob_publication_data/years/2019/preprocessed/2019_d/62.csv',
  'bob_publication_data/years/2019/preprocessed/2019_h/11.csv',
  'bob_publication_data/years/2019/p

## 3. Anchor Table

> **Anchor Table**
>
> The `honey_daily` table is treated as the raw DuckDB anchor table. It preserves the imported daily measurement files with minimal changes. Downstream cleaning, type conversion, feature engineering, and modeling steps should build from this table rather than modifying it directly.

In [17]:
con.execute("""
SHOW TABLES;
""").df()

,name
0,honey_daily


In [18]:
daily_measurement_folder = project_root / "Data" / "Daily_Measurements"
csv_pattern = str(daily_measurement_folder / "**" / "*.csv")

daily_measurement_csvs = list(daily_measurement_folder.rglob("*.csv"))

print(f"Daily measurement folder: {daily_measurement_folder}")
print(f"Daily measurement CSV files: {len(daily_measurement_csvs)}")
print(csv_pattern)

Daily measurement folder: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\Daily_Measurements
Daily measurement CSV files: 453
C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\Daily_Measurements\**\*.csv


In [19]:
# Check whether the raw anchor table already exists
existing_tables = con.execute("SHOW TABLES;").df()

if "honey_daily" in existing_tables["name"].values:
    print("Using existing anchor table: honey_daily")
else:
    print("Creating anchor table: honey_daily")

    con.execute(f"""
    CREATE TABLE honey_daily AS
    SELECT *
    FROM read_csv_auto(
        '{csv_pattern}',
        filename = true,
        union_by_name = true,
        all_varchar = true
    );
    """)

    print("Created anchor table: honey_daily")

Using existing anchor table: honey_daily


In [20]:
# Extract only daily measurement CSVs
with zipfile.ZipFile(zip_path) as z:
    for file in measurement_files:
        z.extract(file, path=daily_measurement_folder)

print(f"Extracted {len(measurement_files)} daily measurement CSV files.")

Extracted 453 daily measurement CSV files.


In [21]:
daily_measurement_csvs = list(daily_measurement_folder.rglob("*.csv"))

len(daily_measurement_csvs), daily_measurement_csvs[:5]

(453,
 [WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data/Daily_Measurements/bob_publication_data/years/2019/preprocessed/2019_d/11.csv'),
  WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data/Daily_Measurements/bob_publication_data/years/2019/preprocessed/2019_d/22.csv'),
  WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data/Daily_Measurements/bob_publication_data/years/2019/preprocessed/2019_d/25.csv'),
  WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data/Daily_Measurements/bob_publication_data/years/2019/preprocessed/2019_d/26.csv'),
  WindowsPath('C:/Users/sfnor/Miniconda3/envs/pandas_book/honey-yield-predictive-model/Data/Daily_Measurements/bob_publication_data/years/2019/preprocessed/2019_d/27.csv')])

> **Why load all columns as `VARCHAR`?**
>
> The daily measurement files contain schema differences and inconsistent data types. Loading all columns as text prevents import failures caused by mixed numeric, Boolean, and missing values. Data types will be validated and converted during the transformation stage after all files have been successfully loaded into DuckDB.

In [22]:
con.execute("""
SHOW TABLES;
""").df()

,name
0,honey_daily


In [23]:
con.execute("""
SELECT COUNT(*) AS row_count
FROM honey_daily;
""").df()

,row_count
0,36697106


In [24]:
con.execute("""
SELECT *
FROM honey_daily
LIMIT 10;
""").df()

,column00,X.1,time,X,t_i_1,t_i_2,t_i_3,t_i_4,t_i_5,t_o,...,died.next,died.last.dif,died.next.dif,swarming.last,swarming.next,swarming.last.dif,swarming.next.dif,lon,lat,filename
0,1,1,2019-06-24,251869,30.1617157794677,30.385575095057,26.3360266159696,26.0898288973384,NA,20.4259743346008,...,NA,NA,NA,NA,NA,NA,339.55,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
1,2,2,2019-06-25,252720.5,31.2451388888889,32.2414930555556,29.9884114583333,29.8963541666667,NA,25.7718967013889,...,NA,NA,NA,NA,NA,NA,338.958680555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
2,3,3,2019-06-26,254160.5,30.2679036458333,30.3205295138889,27.1550998263889,26.8098958333333,NA,21.6816840277778,...,NA,NA,NA,NA,NA,NA,337.958680555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
3,4,4,2019-06-27,255550.437213566,26.2060895967003,25.5157252520623,18.6211617781852,18.2064906049496,NA,13.6424725022915,...,NA,NA,NA,NA,NA,NA,336.993446379468,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
4,5,5,2019-06-28,256601,26.8146167557932,26.2207553475936,15.9831773618538,18.8590129233512,NA,13.7718917112299,...,NA,NA,NA,NA,NA,NA,336.263888888889,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
5,6,6,2019-07-02,262850,28.2874087591241,28.0590784671533,24.6019616788321,24.1439324817518,NA,19.0971715328467,...,NA,NA,NA,NA,NA,NA,331.924305555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
6,7,7,2019-07-17,284713.114391144,25.7999153325123,25.1900015394089,22.6236530172414,22.3527555418719,NA,18.4852601600985,...,NA,NA,NA,NA,NA,NA,316.741587228372,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
7,8,8,2019-07-18,285840.5,25.4922960069444,24.67265625,22.0047092013889,21.7590494791667,NA,19.3501953125,...,NA,NA,NA,NA,NA,NA,315.958680555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
8,9,9,2019-07-19,287280.5,26.7734592013889,26.2806423611111,23.8123480902778,23.5030381944444,NA,20.2501085069444,...,NA,NA,NA,NA,NA,NA,314.958680555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...
9,10,10,2019-07-20,288720.5,25.0897786458333,24.2613498263889,21.5822482638889,21.3791883680556,NA,18.82421875,...,NA,NA,NA,NA,NA,NA,313.958680555556,8.8,53.1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...


In [25]:
con.execute("""
DESCRIBE honey_daily;
""").df()

,column_name,column_type,null,key,default,extra
0,column00,VARCHAR,YES,None,None,None
1,X.1,VARCHAR,YES,None,None,None
2,time,VARCHAR,YES,None,None,None
3,X,VARCHAR,YES,None,None,None
4,t_i_1,VARCHAR,YES,None,None,None
5,t_i_2,VARCHAR,YES,None,None,None
6,t_i_3,VARCHAR,YES,None,None,None
7,t_i_4,VARCHAR,YES,None,None,None
8,t_i_5,VARCHAR,YES,None,None,None
9,t_o,VARCHAR,YES,None,None,None


# 4. Explore the Raw Data

In [26]:
con.execute("""
SELECT COUNT(*) AS row_count
FROM honey_daily;
""").df()

,row_count
0,36697106


In [27]:
con.execute("""
SELECT COUNT(DISTINCT filename) AS source_file_count
FROM honey_daily;
""").df()

,source_file_count
0,453


In [28]:
con.execute("""
SELECT
    MIN(time) AS min_time,
    MAX(time) AS max_time
FROM honey_daily;
""").df()

,min_time,max_time
0,2019-06-24,2022-12-31 23:58:00


In [29]:
con.execute("""
SELECT
    year,
    COUNT(*) AS row_count
FROM honey_daily
GROUP BY year
ORDER BY year;
""").df()

,year,row_count
0,2019,1099939
1,2020,12036900
2,2021,12084504
3,2022,11475763


```2019 contains substantially fewer records because data collection began during the year. The remaining years contain approximately 11–12 million observations each, providing relatively balanced longitudinal coverage.```

In [30]:
con.execute("""
SELECT
    filename,
    COUNT(*) AS row_count
FROM honey_daily
GROUP BY filename
ORDER BY row_count DESC
LIMIT 10;
""").df()

,filename,row_count
0,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,486444
1,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,484745
2,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,484359
3,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,483693
4,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,480278
5,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,476885
6,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,475533
7,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,474407
8,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,471659
9,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,471306


In [31]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS time_nulls,
    SUM(CASE WHEN weight_kg IS NULL THEN 1 ELSE 0 END) AS weight_nulls,
    SUM(CASE WHEN t_i_1 IS NULL THEN 1 ELSE 0 END) AS internal_temp1_nulls,
    SUM(CASE WHEN t_o IS NULL THEN 1 ELSE 0 END) AS outside_temp_nulls,
    SUM(CASE WHEN h IS NULL THEN 1 ELSE 0 END) AS humidity_nulls,
    SUM(CASE WHEN p IS NULL THEN 1 ELSE 0 END) AS pressure_nulls,
    SUM(CASE WHEN lon IS NULL THEN 1 ELSE 0 END) AS longitude_nulls,
    SUM(CASE WHEN lat IS NULL THEN 1 ELSE 0 END) AS latitude_nulls

FROM honey_daily;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,time_nulls,weight_nulls,internal_temp1_nulls,outside_temp_nulls,humidity_nulls,pressure_nulls,longitude_nulls,latitude_nulls
0,36697106,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 5. Data Quality Assessment

In [32]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN weight_kg IS NULL THEN 1 ELSE 0 END) AS weight_nulls,
    SUM(CASE WHEN weight_kg = 'NA' THEN 1 ELSE 0 END) AS weight_na,
    SUM(CASE WHEN weight_kg = '' THEN 1 ELSE 0 END) AS weight_blank

FROM honey_daily;
""").df()

,total_rows,weight_nulls,weight_na,weight_blank
0,36697106,0.0,28848.0,0.0


>Observation
>
>The dataset does not use SQL NULL values for missing observations. Instead, missing measurements are represented by the string "NA". During the transformation stage, these values will be converted to proper SQL NULLs before numeric type conversion.

In [33]:
con.execute("""
SELECT
    SUM(CASE WHEN weight_kg = 'NA' THEN 1 ELSE 0 END) AS weight_na,
    SUM(CASE WHEN t_i_1 = 'NA' THEN 1 ELSE 0 END) AS internal_temp_na,
    SUM(CASE WHEN t_o = 'NA' THEN 1 ELSE 0 END) AS outside_temp_na,
    SUM(CASE WHEN h = 'NA' THEN 1 ELSE 0 END) AS humidity_na,
    SUM(CASE WHEN p = 'NA' THEN 1 ELSE 0 END) AS pressure_na,
    SUM(CASE WHEN lon = 'NA' THEN 1 ELSE 0 END) AS longitude_na,
    SUM(CASE WHEN lat = 'NA' THEN 1 ELSE 0 END) AS latitude_na
FROM honey_daily;
""").df()

,weight_na,internal_temp_na,outside_temp_na,humidity_na,pressure_na,longitude_na,latitude_na
0,28848.0,9287528.0,8389058.0,14386423.0,12005893.0,0.0,0.0


# 6. Transform the Data

# 7. Feature Engineering

# 8. Build Modeling Dataset

# 9. Predictive Modeling